<div align="center">

# ASL

</div>

In [1]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import torch

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

DEVICE = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'


dataset = datasets.ImageFolder("data/asl_alphabet", transform=transform)

loader = DataLoader(dataset, batch_size=32, shuffle=True)

images, labels = next(iter(loader))

images = images[:10]
labels = labels[:10]

fig, axes = plt.subplots(2, 5, figsize=(12, 6))
for i, ax in enumerate(axes.flatten()):
    img = images[i].permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(f"Label: {labels[i].item()}")
    ax.axis('off')

AttributeError: module 'torch' has no attribute 'accelerator'

Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

<div align="center" style="color: purple">

## **Download dataset**

</div>

In [ ]:
!kaggle datasets download datamunge/sign-language-mnist

In [ ]:
# import kagglehub
# import os

# # Define the path for the dataset
# dataset_path = "."
# dataset_zip_name = "sign-language-mnist.zip"
# dataset_zip_path = os.path.join(dataset_path, dataset_zip_name)

# print(f"Current working directory: {os.getcwd()}")

# # Check if the dataset zip file already exists
# if os.path.exists(dataset_zip_path):
#     print(f"Dataset '{dataset_zip_name}' already downloaded in the current directory.")
#     path = dataset_zip_path
# else:
#     print("Downloading dataset...")
#     # Download latest version to the current directory
#     path = kagglehub.dataset_download(
#         "datamunge/sign-language-mnist",
#         path=dataset_path
#     )
#     print("Dataset downloaded successfully.")

# print("Path to dataset files:", path)

## Getting the data

In [ ]:
import zipfile

with zipfile.ZipFile("sign-language-mnist.zip", 'r') as zip_ref:
    zip_ref.extractall("data/asl_mnist")

In [ ]:
import pandas as pd

train_path = "data/asl_mnist/sign_mnist_train.csv"
test_path = "data/asl_mnist/sign_mnist_test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

In [ ]:
import matplotlib.pyplot as plt

label_counts = train_df["label"].value_counts().sort_index()

plt.figure(figsize=(12, 5))
plt.bar(label_counts.index, label_counts.values)
plt.xlabel("Label")
plt.ylabel("Number of images")
plt.title("Class distribution in training set")
plt.show()

In [ ]:
#  

import pandas as pd
import torch
from torch.utils.data import Dataset

class SignLanguageMNIST(Dataset):
    def __init__(self, csv_file):
        data = pd.read_csv(csv_file)
        self.labels = data.iloc[:, 0].values
        self.images = data.iloc[:, 1:].values.reshape(-1, 28, 28)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = torch.tensor(self.images[idx], dtype=torch.float32).unsqueeze(0) / 255.0
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, label

In [ ]:
import torch
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

